In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr
from scipy.spatial.distance import euclidean, cdist

import sys

sys.path.append("..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from skimage.filters import gaussian, median

INFO:Constants:Logging to file: /home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/logs/Constants_20251105_165456.log


In [2]:
# data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/old/'
data_folder = "../Camera_Calibrations/Ximea_Camera/"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
image_size = 12
camera_parameters = {}
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ["B", "G", "R"]
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

In [3]:
import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [4]:
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r405-488-561-635-t1-25x36"
nilered_filter = "semrock-ff01-650-200-25"
shortpass_filter = "semrock-bsp01-785r"

filters = [notch_filter, dichroic_mirror, shortpass_filter]
filter_spectra = S_F.get_dye_or_filter_data(
    names=filters, wavelength=wavelength, dye_or_filter=False
)

In [5]:
single_molecule_dyes = np.array(
    [
        ["ATTO 488", 2073],
        ["Alexa Fluor 488", 2811],
        ["CF488A", 3879],
        #["ATTO 594", 2000],
        ["Cy3B", 23195],
        ["ATTO 565", 11600],
        ["Janelia Fluor JF585-HaloTag conjugate", 2429],
        ["CF568", 13388],
        ["Cy5", 7801],
        ["Cy5B", 17090],
        ["ATTO 643", 23327],
        ["ATTO 647N", 18448],
        ["ATTO 655", 8273],
        ["CF640R", 12024],
        ["CF660R", 10399],
        ["abberior STAR 635", 12731],
        ["Janelia Fluor JF646-HaloTag conjugate", 14440],
        ["Alexa Fluor 647", 10348],
        ["Cy2", 6241],
        ["Cy3", 11022],
        ["Tetramethylrhodamine (TAMRA, TRITC)", 4884],
        ["Cy3.5", 4968],
        ["ATTO 647", 1526],
        ["ATTO 680", 1656],
        ["Cy5.5", 6337],
        ["Cy7", 852],
        ["Alexa Fluor 750", 703],
        ["ATTO 740", 779],
        ["Alexa Fluor 790", 740],
    ],
    dtype="object",
)

In [11]:
potential_dyes = [
    "Cy3B",
    #"ATTO 594",
    "CF550R",
    "Cy5B",
    "Alexa Fluor 488",
    "ATTO 488",
    "ATTO 643",
    "ATTO 647N",
    "ATTO 655",
    "Alexa Fluor 647",
    "CF640R",
    "CF660R",
    "abberior STAR 635",
    "ATTO 620",
    "ATTO 565",
    "CF568",
    "Janelia Fluor JF646-HaloTag conjugate",
    "ATTO 680",
    
]

In [14]:
import numpy as np
from src import Multicolour_Simulation_Functions
from src import SpectralFunctions

# Select optimal 5 dyes
result = MSF.optimal_dye_selector_simulated(
    potential_dyes=potential_dyes,
    single_molecule_dyes=single_molecule_dyes,
    filters=filters,
    smoothing_function=smoothing_function,
    camera_parameters=camera_parameters,
    wavelength=wavelength,
    n_dyes_desired=5,
    min_photons_per_100ms=500,
    n_simulations=10000,
    exhaustive_search=True,  # Use greedy for speed
    background_photons=50,
    verbose=True
)

print(f"\nOptimal dyes: {result['selected_dyes']}")
print(f"Classification accuracy: {result['overall_accuracy']:.1%}")


OPTIMAL DYE SELECTION VIA SIMULATION

Step 1: Filtering dyes (min 500 photons/100ms)...


  17 candidates -> 14 viable dyes
  Rejected: {'ATTO 620', 'ATTO 680', 'CF550R'}

Step 2-3: Simulating 10000 molecules per dye...
  Simulating Cy3B (23195 source / 4878 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.554, 0.413)
    Std  (A_R, A_G): (0.008, 0.008)
  Simulating Cy5B (17090 source / 3594 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.710, 0.222)
    Std  (A_R, A_G): (0.009, 0.008)
  Simulating Alexa Fluor 488 (2811 source / 591 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.090, 0.745)
    Std  (A_R, A_G): (0.019, 0.025)
  Simulating ATTO 488 (2073 source / 436 detector photons)...
    Fit success: 10000/10000 (100.0%)                                           ?, ?it/s]
    Mean (A_R, A_G): (0.100, 0.745)
    Std